<a href="https://colab.research.google.com/github/joae1234/movie_llm/blob/main/back_end.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Coleta de dados
É feita a coleta de 1000 filmes com gêneros específicos do TMDB. O código atual busca um número fixo de páginas. Preciso modificar o loop de busca para continuar até que 1000 filmes com os gêneros alvo sejam coletados, adicionando também uma salvaguarda contra loops infinitos.

In [1]:
import requests
import pandas as pd
from google.colab import userdata
userdata.get('movies_key')

API_KEY = userdata.get('movies_key')
BASE_URL = "https://api.themoviedb.org/3"

def get_movies(page=1, language="pt-BR"):
    url = f"{BASE_URL}/movie/popular?api_key={API_KEY}&language={language}&page={page}"
    response = requests.get(url)
    return response.json()

def get_movie_details(movie_id, language="pt-BR"):
    url = f"{BASE_URL}/movie/{movie_id}?api_key={API_KEY}&language={language}"
    response = requests.get(url)
    return response.json()

# Define the target genres
target_genres = ["Ação", "Comédia", "Drama", "Ficção científica", "Terror"]

# Function to get the single genre based on the target list
def get_single_genre(genres):
    if not genres:
        return None
    for genre in genres:
        if genre["name"] in target_genres:
            return genre["name"]
    return None # Return None if no target genre is found

# Collect movies until 1000 with target genres are found
movies = []
page = 1
max_pages_to_check = 100 # Safeguard against infinite loops

while len([m for m in movies if m['genre'] is not None]) < 1000 and page <= max_pages_to_check:
    data = get_movies(page)
    if not data or not data.get("results"):
        break # Stop if no more data is returned

    for m in data["results"]:
        details = get_movie_details(m["id"])
        genres = details.get("genres", [])
        single_genre = get_single_genre(genres)
        if single_genre in target_genres: # Only add movies with target genres
             movies.append({
                "id": m["id"],
                "title": m["title"],
                "overview": m["overview"],
                "genre": single_genre # Store the single genre
            })

    page += 1
    # Optional: Add a small delay here if needed to avoid hitting API rate limits
    # import time
    # time.sleep(0.1)


df = pd.DataFrame(movies)
display(df.head())
print(f"Collected {len(df)} movies with target genres.")


,id,title,overview,genre
0,507244,Caçadores do Fim do Mundo,Uma década após uma tempestade solar devastar ...,Ficção científica
1,1156594,Nossa Culpa,O casamento de Jenna e Lion marca o tão espera...,Drama
2,755898,A Guerra dos Mundos,Will Radford é um renomado analista de ciberse...,Ficção científica
3,1305717,Hunting Grounds,Uma mãe que foge do marido ligado à máfia enco...,Ação
4,1306525,O Elixir,Um elixir desperta os mortos-vivos em uma vila...,Terror


Collected 1013 movies with target genres.



Coletei com sucesso mais de 1000 filmes com os gêneros alvo. O próximo passo, de acordo com a descrição geral da tarefa, é garantir uma distribuição igual de filmes entre os gêneros alvo. Preciso amostrar os filmes coletados para ter um número igual para cada gênero alvo, até um máximo de 200 por gênero para atingir o objetivo de 1000 filmes no total (200 * 5 gêneros).

In [2]:
# Ensure an equal distribution of movies across the target genres
balanced_movies = []
max_per_genre = 1000 // len(target_genres) # Aim for equal distribution

for genre in target_genres:
    genre_movies = df[df['genre'] == genre].sample(min(len(df[df['genre'] == genre]), max_per_genre), random_state=42)
    balanced_movies.append(genre_movies)

df_balanced = pd.concat(balanced_movies).sample(frac=1, random_state=42).reset_index(drop=True)

display(df_balanced.head())
print(f"Collected {len(df_balanced)} movies with balanced genre distribution.")
print("Genre distribution in balanced DataFrame:")
print(df_balanced['genre'].value_counts())

,id,title,overview,genre
0,83533,Avatar: Fogo e Cinzas,Após a devastadora guerra contra a RDA e a per...,Ficção científica
1,949423,Pearl: Uma História de Origem 'X',"Presa na fazenda isolada de sua família, Pearl...",Terror
2,109418,Gente Grande 2,Lenny Feder e sua família se mudam para sua ci...,Comédia
3,1419229,Silent Zone,"Em um mundo devastado por zumbis, uma adolesce...",Terror
4,505642,Pantera Negra: Wakanda para Sempre,"A rainha Ramonda, Shuri, M'Baku, Okoye e as po...",Ação


Collected 836 movies with balanced genre distribution.
Genre distribution in balanced DataFrame:
genre
Comédia              200
Drama                200
Ação                 200
Terror               165
Ficção científica     71
Name: count, dtype: int64


In [3]:
import requests
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

Coletei e equilibrei com sucesso o conjunto de dados por gênero. O próximo passo é preparar os dados para treinar o modelo, o que envolve codificar os rótulos e dividir os dados em conjuntos de treinamento e teste. Também inicializarei o tokenizador e prepararei os conjuntos de dados para o modelo transformer.


In [4]:
le = LabelEncoder()
df_balanced["label"] = le.fit_transform(df_balanced["genre"])

# treino e teste
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_balanced["overview"].tolist(), df_balanced["label"].tolist(), test_size=0.2, random_state=42
)

# tokenizer
checkpoint = "distilbert-base-multilingual-cased"  # suporta português
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
test_dataset  = Dataset.from_dict({"text": test_texts, "label": test_labels})

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/668 [00:00<?, ? examples/s]

Map:   0%|          | 0/168 [00:00<?, ? examples/s]

Os dados foram preparados e tokinizados. O proximo passo é inicar o modelo para a sequência de classificação usando os checkpoint já treinados e o numero de labels unicas para um dataset balanceado

In [5]:
num_labels = len(le.classes_)
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels
)

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


O modelo foi inicializado. O próximo passo é treinar o modelo usando os conjuntos de dados de treinamento e teste preparados e os argumentos e métricas de treinamento definidos.